# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [1]:
import pandas as pd

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [2]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']   #Los nombre de las columnas según el enunciado
df = pd.read_csv(url, header=None, names=names)   #Pongo al df los nombre de las columnas
df.head()


,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.info()   #Veo 768 entradas, ningún nulos y todos los datos numéricos

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.describe()

,preg,plas,pres,skin,test,mass,pedi,age,class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [5]:
#Veo que tengo valores de 0 (en min) en algunas features, lo cual no es 'normal'

In [6]:
#Cantidad absoluta de ceros por columna
ceros = (df == 0).sum()
print(ceros)

preg     111
plas       5
pres      35
skin     227
test     374
mass      11
pedi       0
age        0
class    500
dtype: int64


In [7]:
#Podría sustituir estos valores con la media?¿? o falsearían los resultados?¿?

### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [8]:
X = df.drop('class', axis=1)   #Me dicen que no divida en train/test, así que mis features serían todas las columnas menos el target ('class')
y =df['class']

In [9]:
#Establezco mi semilla
seed = 42

In [10]:
#Configuro la validación cruzada (Cross Validation) con 10 kfolds, ed, entrenaré el modelo 10 veces (cada vuelta entrena con 9 grupos y valida con 1)

from sklearn.model_selection import KFold

kfold = KFold(n_splits=10, shuffle=True, random_state=seed)   #shuffle=True para que mezcle los datos antes de partirlos

In [11]:
#Creo el modelo de Bagging con 100 DecisionTreeClassifier

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

estimator = DecisionTreeClassifier(random_state=seed)
model_bag_clf = BaggingClassifier(estimator=estimator, n_estimators=100, random_state=seed)

In [12]:
#Entreno y evalúo mi modelo

from sklearn.model_selection import cross_val_score

results_bag = cross_val_score(estimator=model_bag_clf, X=X, y=y, cv=kfold)   #cv define cómo dividir los datos, ed el kfold que creé
print(f'Accuracy de cada uno de los 10 folds de la CV: {results_bag}\n')

accuracy_bag = results_bag.mean()   #Saco la media de los 10 resultados
print(f'Bagging Accuracy (media): {accuracy_bag:.3f}')   #:.3f para que me devuelva 3 decimales

Accuracy de cada uno de los 10 folds de la CV: [0.72727273 0.75324675 0.7012987  0.85714286 0.79220779 0.71428571
 0.77922078 0.76623377 0.67105263 0.80263158]

Bagging Accuracy (media): 0.756


### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [13]:
#Creo el modelo de RF con 100 árboles y max de features de 3

from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators=100, max_features=3, random_state=seed)

In [14]:
#Entreno y evalúo mi modelo

results_rf = cross_val_score(estimator=model_rf, X=X, y=y, cv=kfold)   #cv define cómo dividir los datos, mantengo el kfold que creé al ppio
print(f'Accuracy de cada uno de los 10 folds de la CV: {results_rf}\n')

accuracy_rf = results_rf.mean()   #Saco la media de los 10 resultados
print(f'Random Forest Accuracy (media): {accuracy_rf:.3f}')   #:.3f para que me devuelva 3 decimales

Accuracy de cada uno de los 10 folds de la CV: [0.72727273 0.77922078 0.74025974 0.85714286 0.84415584 0.72727273
 0.79220779 0.74025974 0.67105263 0.81578947]

Random Forest Accuracy (media): 0.769


### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [15]:
#Creo el modelo de AB con 30 árboles

from sklearn.ensemble import AdaBoostClassifier

model_ab = AdaBoostClassifier(n_estimators=30, random_state=seed)

In [16]:
#Entreno y evalúo mi modelo

results_ab = cross_val_score(estimator=model_ab, X=X, y=y, cv=kfold)   #cv define cómo dividir los datos, mantengo el kfold que creé al ppio
print(f'Accuracy de cada uno de los 10 folds de la CV: {results_ab}\n')

accuracy_ab = results_ab.mean()   #Saco la media de los 10 resultados
print(f'AdaBoost Accuracy (media): {accuracy_ab:.3f}')   #:.3f para que me devuelva 3 decimales

Accuracy de cada uno de los 10 folds de la CV: [0.74025974 0.79220779 0.72727273 0.77922078 0.83116883 0.67532468
 0.79220779 0.77922078 0.68421053 0.81578947]

AdaBoost Accuracy (media): 0.762


### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [17]:
#Creo el modelo de GB con 100 estimadores

from sklearn.ensemble import GradientBoostingClassifier

model_gb = GradientBoostingClassifier(n_estimators=100, random_state=seed)

In [18]:
#Entreno y evalúo mi modelo

results_gb = cross_val_score(estimator=model_gb, X=X, y=y, cv=kfold)   #cv define cómo dividir los datos, mantengo el kfold que creé al ppio
print(f'Accuracy de cada uno de los 10 folds de la CV: {results_gb}\n')

accuracy_gb = results_gb.mean()   #Saco la media de los 10 resultados
print(f'GradientBoosting Accuracy (media): {accuracy_gb:.3f}')   #:.3f para que me devuelva 3 decimales

Accuracy de cada uno de los 10 folds de la CV: [0.72727273 0.79220779 0.74025974 0.85714286 0.81818182 0.72727273
 0.81818182 0.74025974 0.65789474 0.78947368]

GradientBoosting Accuracy (media): 0.767


### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [19]:
!pip install xgboost


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
#Creo el modelo de XGB con 100 estimadores

from xgboost import XGBClassifier

model_xgb = XGBClassifier(n_estimators=100, random_state=seed)

In [21]:
#Entreno y evalúo mi modelo

results_xgb = cross_val_score(estimator=model_xgb, X=X, y=y, cv=kfold)   #cv define cómo dividir los datos, mantengo el kfold que creé al ppio
print(f'Accuracy de cada uno de los 10 folds de la CV: {results_xgb}\n')

accuracy_xgb = results_xgb.mean()   #Saco la media de los 10 resultados
print(f'XGBoost Accuracy (media): {accuracy_xgb:.3f}')   #:.3f para que me devuelva 3 decimales

Accuracy de cada uno de los 10 folds de la CV: [0.71428571 0.67532468 0.71428571 0.80519481 0.77922078 0.68831169
 0.77922078 0.72727273 0.69736842 0.80263158]

XGBoost Accuracy (media): 0.738


### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [22]:
resultados = pd.DataFrame({
    'Algoritnmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy': [accuracy_bag, accuracy_rf, accuracy_ab, accuracy_gb, accuracy_xgb]
})

resultados_orden = resultados.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
resultados_orden


,Algoritnmo,Accuracy
0,Random Forest,0.769463
1,GradientBoosting,0.766815
2,AdaBoost,0.761688
3,Bagging,0.756459
4,XGBoost,0.738312


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [23]:
#Divido entre train/test. Sigo la consigna habitual de 80/20

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

In [24]:
#Defino modelos y rejillas

from sklearn.model_selection import GridSearchCV

modelos = {   #Diccionario con los modelos base
    'Bagging': model_bag_clf,
    'Random Forest': model_rf,
    'AdaBoost': model_ab,
    'GradientBoosting': model_gb,
    'XGBoost': model_xgb
}

grids = {   #Diccionario con los grids (hiperparámetros a probar). 
    'Bagging': {'n_estimators': [50, 100, 200], 'max_samples': [0.5, 0.8, 1.0], 'max_features': [0.5, 0.8, 1.0]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, 30, None], 'max_features': [0.5, 0.8, 1.0, 'sqrt']},
    'AdaBoost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5, 1.0]},
    'GradientBoosting': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5, 1.0], 'max_depth': [1, 2, 3]},
    'XGBoost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5, 1.0], 'max_depth': [1, 2, 3]}
}


In [25]:
#Entreno con GridSearchCV y evalúo

from sklearn.metrics import accuracy_score

resultados_grid = []

for nombre in modelos:
    print(f"Optimizando {nombre}...")
    grid = GridSearchCV(
        estimator=modelos[nombre],
        param_grid=grids[nombre],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    y_pred = grid.best_estimator_.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred)
    
    resultados_grid.append({
        'Algoritmo': nombre,
        'Accuracy Train (CV)': round(grid.best_score_, 4),
        'Accuracy Test': round(acc_test, 4),
        'Mejores Parámetros': grid.best_params_
    })

Optimizando Bagging...
Optimizando Random Forest...
Optimizando AdaBoost...
Optimizando GradientBoosting...
Optimizando XGBoost...


In [26]:
df_final = pd.DataFrame(resultados_grid).sort_values(by='Accuracy Test', ascending=False).reset_index(drop=True)
df_final

,Algoritmo,Accuracy Train (CV),Accuracy Test,Mejores Parámetros
0,XGBoost,0.7802,0.7597,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti..."
1,GradientBoosting,0.7786,0.7597,"{'learning_rate': 0.5, 'max_depth': 1, 'n_esti..."
2,Bagging,0.7867,0.7532,"{'max_features': 1.0, 'max_samples': 0.5, 'n_e..."
3,AdaBoost,0.7802,0.7403,"{'learning_rate': 1.0, 'n_estimators': 100}"
4,Random Forest,0.7769,0.7078,"{'max_depth': 20, 'max_features': 'sqrt', 'n_e..."


### 10. Conclusiones finales

* **Cambio de liderazgo tras optimizar:**  
En la primera fase (validación cruzada sobre todo el dataset sin optimizar), **Random Forest** obtuvo el mejor *accuracy* medio (`0.7695`), mientras que **XGBoost** quedó en última posición (`0.7383`).  
Tras aplicar `GridSearchCV` y evaluar en *test*, los papeles se invirtieron: **XGBoost** y **Gradient Boosting** empataron en el primer puesto con un *accuracy* en *test* de **`0.7597`**.